# Track 06 — LangGraph Stateful RAG: Cyclic State Machines, CRAG, Self-RAG & HITL

Welcome to Track 06 of the **RAG Engineering Academy**.

In linear RAG pipelines (such as standard LangChain LCEL chains), execution is a unidirectional Directed Acyclic Graph (DAG):
$$\text{Query} \longrightarrow \text{Retriever} \longrightarrow \text{Context} \longrightarrow \text{LLM} \longrightarrow \text{Answer}$$

While straightforward, unidirectional DAGs break down in real-world environments when:
1. **Retrieved context is irrelevant or noisy**: A linear chain cannot reject poor context—it forwards irrelevant data to the LLM, causing hallucinations.
2. **The user query requires iterative refinement**: If initial retrieval fails, linear pipelines cannot reformulate the query and re-retrieve.
3. **Execution requires human approval**: Production enterprise systems often require human sign-off before returning high-stakes outputs.

### What LangGraph Solves:
LangGraph models LLM workflows as **state machines** and **cyclic computation graphs**:
- **StateGraph**: A shared, versioned state schema (`TypedDict`) passed across nodes.
- **Corrective RAG (CRAG)**: Evaluates retrieved document relevance and rewrites queries in a cycle when relevance fails.
- **Self-RAG**: Evaluates generated text for hallucination and answer utility before returning it.
- **Loop Prevention & Budgets**: Hard-coded limits (`max_retries`) preventing infinite recursion and token burn.
- **State Persistence & Time Travel**: `MemorySaver` checkpointer for thread isolation and checkpoint inspection.
- **Human-in-the-Loop (HITL)**: Execution breakpoints (`interrupt_before`) allowing human review and state modification.

![LangGraph CRAG Architecture](../../assets/diagrams/track06_langgraph_crag_architecture.png)

### 1. Environment Verification and Imports

We start by verifying that `langgraph` and core project modules are accessible.
We also load environment variables via `common.config`.

Observe the imports from `langgraph.graph` (`StateGraph`, `START`, `END`) and `langgraph.checkpoint.memory` (`MemorySaver`).

In [ ]:
import os
import sys
from pathlib import Path
from typing import Dict, Any, List, Optional, Literal, TypedDict

# Ensure repository root is on sys.path
root_dir = Path.cwd().resolve().parent.parent if Path.cwd().name == '06-langgraph-rag' else Path.cwd().resolve()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import langgraph
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

from common.config import settings

print("LangGraph successfully loaded.")
print(f"Groq API Key Configured: {'Yes' if settings.groq_api_key else 'No (Deterministic Offline Fallback Active)'}")

### 2. Defining Typed Graph State

In LangGraph, state is a shared dictionary that flows between nodes. Each node receives the current state, executes its discrete logic, and returns a partial dictionary of **state updates**.

Here we inspect `RAGGraphState` defined in `common.langgraph_rag`:
- `question`: The active query string (may be rewritten across iterations).
- `original_question`: The original user query preserved for reference and evaluation.
- `documents`: List of retrieved documents that pass relevance grading.
- `generation`: The synthesized response string.
- `loop_count`: An execution budget counter tracking how many times the graph has cycled.
- `max_retries`: The maximum allowable retry cycles before routing to safe refusal.
- `documents_status`: Categorical status (`"relevant"`, `"irrelevant"`, `"exhausted"`).
- `trace`: An audit list recording the exact sequence of node transitions.

![LangGraph State Machine](../../assets/diagrams/track06_state_machine_cyclical_flow.png)

In [ ]:
from common.langgraph_rag import DocumentDict, RAGGraphState, create_initial_state

# Demonstrate initial state creation
sample_state = create_initial_state(
    question="How do cyclic state graphs handle document relevance failure?",
    max_retries=2
)

print("Sample Pristine Graph State:")
for k, v in sample_state.items():
    print(f"  {k:20s}: {v}")

### 3. Offline-First Deterministic Engines

To adhere to our educational principles (100% offline-executable by default, zero mandatory paid APIs), we instantiate deterministic engines:
- `DeterministicGraphRetriever`: Performs lexical and keyword overlap matching over a local corpus.
- `DocumentRelevanceGrader`: Evaluates semantic overlap between retrieved documents and the query, filtering out noise.
- `GraphQueryTransformer`: Systematically enriches queries with domain keywords to repair failed retrievals.
- `GraphAnswerGenerator`: Generates grounded answers referencing source documents, or produces a polite refusal if context is missing.

In [ ]:
from common.langgraph_rag import (
    DeterministicGraphRetriever,
    DocumentRelevanceGrader,
    GraphQueryTransformer,
    GraphAnswerGenerator,
    SelfRAGHallucinationGrader,
    SelfRAGAnswerGrader
)

retriever = DeterministicGraphRetriever()
grader = DocumentRelevanceGrader(overlap_threshold=1)
transformer = GraphQueryTransformer()
generator = GraphAnswerGenerator()

# Quick test of offline retriever
query = "What is RAG?"
retrieved = retriever.retrieve(query, top_k=2)
print(f"Query: '{query}' -> Retrieved {len(retrieved)} documents:")
for i, doc in enumerate(retrieved, 1):
    print(f"  [{i}] {doc['page_content'][:80]}...")

### 4. Node Factories and Edge Routers

A node in LangGraph is a Python callable: `node(state) -> dict`.

We use factory functions to bind engines to node functions:
- `make_retrieve_node(retriever)`: Fetches candidates and appends `'retrieve'` to trace.
- `make_grade_documents_node(grader)`: Evaluates documents, sets status to `'relevant'` or `'irrelevant'`.
- `make_transform_query_node(transformer)`: Rewrites query, increments `loop_count`.
- `make_generate_node(generator)`: Produces grounded answer.
- `fallback_node`: Refusal fallback triggered when budget is exhausted.

The conditional edge router `decide_crag_path(state)` determines the next node:
- If documents are relevant: route to `'generate'`.
- If documents are irrelevant and `loop_count < max_retries`: route to `'transform_query'`.
- If budget exhausted (`loop_count >= max_retries`): route to `'fallback'`.

In [ ]:
from common.langgraph_rag import (
    make_retrieve_node,
    make_grade_documents_node,
    make_transform_query_node,
    make_generate_node,
    fallback_node,
    decide_crag_path
)

retrieve_fn = make_retrieve_node(retriever)
grade_fn = make_grade_documents_node(grader)

# Step 1: Execute retrieve_node
step1_updates = retrieve_fn(sample_state)
current_state = {**sample_state, **step1_updates}
print("Trace after retrieve_node:", current_state["trace"])

# Step 2: Execute grade_documents_node
step2_updates = grade_fn(current_state)
current_state = {**current_state, **step2_updates}
print("Trace after grade_documents_node:", current_state["trace"])
print("Routing decision:", decide_crag_path(current_state))

### 5. Building the Corrective RAG (CRAG) StateGraph

We now assemble the nodes into a `StateGraph(RAGGraphState)`.

Notice the cycle:
`grade_documents` --(irrelevant)--> `transform_query` ----> `retrieve` ----> `grade_documents`

This cycle allows the system to autonomously self-correct poor retrieval results before generation!

In [ ]:
from common.langgraph_rag import build_crag_graph

crag_app = build_crag_graph(
    retriever=retriever,
    grader=grader,
    transformer=transformer,
    generator=generator
)

print("CRAG StateGraph successfully compiled!")

### 6. Executing CRAG Across Three Scenarios

We now execute our compiled CRAG graph on three distinct queries:
1. **Direct Hit**: `"Explain chunking strategies and token boundary"` (matches on first pass).
2. **Self-Correcting Cycle**: `"Tell me about cycle mechanisms"` (initial retrieval is insufficient, query transformation triggers, second retrieval succeeds).
3. **Budget Exhaustion**: `"Cosmological redshift and gravitational lensing"` (no matching context exists; graph cycles up to `max_retries` and cleanly routes to safe fallback refusal).

In [ ]:
queries = [
    ("Direct Hit", "Explain chunking strategies and token boundary"),
    ("Cyclical Recovery", "Tell me about cycle mechanisms"),
    ("Budget Exhaustion", "Cosmological redshift and gravitational lensing"),
]

for label, q in queries:
    print(f"\n{'='*70}")
    print(f"SCENARIO: {label}")
    print(f"Input Query: '{q}'")
    state = create_initial_state(q, max_retries=2)
    output = crag_app.invoke(state)
    
    print(f"Execution Trace : {' -> '.join(output['trace'])}")
    print(f"Loop Count      : {output['loop_count']}")
    print(f"Document Status : {output['documents_status']}")
    print(f"Generation Result:")
    print(f"  {output['generation']}")

### 7. Self-RAG: Reflection on Groundedness and Utility

Self-RAG adds post-generation reflection:
1. **Hallucination Grader**: Does the answer contain claims unsupported by the documents?
2. **Answer Utility Grader**: Does the answer actually address the user's inquiry?

If the answer fails reflection, the graph re-enters the retrieval or generation loop instead of emitting incorrect outputs.

In [ ]:
from common.langgraph_rag import build_self_rag_graph

self_rag_app = build_self_rag_graph()

state = create_initial_state("LangGraph stateful cyclic graphs")
output = self_rag_app.invoke(state)

print("Self-RAG Output:")
print(f"Trace: {' -> '.join(output['trace'])}")
print(f"Hallucination Grade: {output['hallucination_grade']}")
print(f"Answer Grade       : {output['answer_grade']}")
print(f"Answer: {output['generation']}")

### 8. Persistence, Thread Isolation, and Human-in-the-Loop (HITL)

Enterprise applications require:
1. **Thread Isolation**: Separate users or conversations must have isolated state.
2. **Checkpointing**: Every node execution is persisted to storage (`MemorySaver` or Postgres).
3. **Breakpoints (HITL)**: Interrupting execution before critical actions so human reviewers can inspect and approve.

Here, we compile the adaptive graph with `interrupt_before=["human_review"]`. Execution will automatically pause at the review gate, waiting for human approval!

In [ ]:
from common.langgraph_rag import build_adaptive_rag_graph

checkpointer = MemorySaver()
adaptive_app = build_adaptive_rag_graph(checkpointer=checkpointer, enable_hitl=True)

# Define thread configuration
config = {"configurable": {"thread_id": "enterprise-session-42"}}

# Run until the breakpoint
state = create_initial_state("LangGraph cyclic checkpointing")
print("Starting graph execution...")
adaptive_app.invoke(state, config=config)

checkpoint_state = adaptive_app.get_state(config)
print(f"Current Node Location : {checkpoint_state.next}")
print(f"Execution Trace so far: {checkpoint_state.values['trace']}")
print(f"Generated Draft: {checkpoint_state.values['generation'][:90]}...")
print("\n[ACTION REQUIRED]: Execution interrupted at human_review gate!")

### 8b. Resuming Execution After Human Approval

A human reviewer inspects the draft, confirms it is factual and compliant, and updates the state with `human_approval: True`.

Calling `app.invoke(None, config)` resumes execution seamlessly from the checkpoint!

In [ ]:
# Human provides approval
print("Human Reviewer approves generated draft.")
adaptive_app.update_state(config, {"human_approval": True})

# Resume execution
final_state = adaptive_app.invoke(None, config=config)

print("\nResumed Execution Result:")
print(f"Final Trace: {' -> '.join(final_state['trace'])}")
print(f"Human Approval: {final_state['human_approval']}")
print(f"Next steps pending: {adaptive_app.get_state(config).next} (Empty = Workflow Completed)")

### 9. Optional Live Groq API Synthesis

If a `GROQ_API_KEY` is configured in your `.env`, we can run synthesis through Groq's Llama-3 model. If not, the deterministic offline engine continues to provide predictable educational outputs.

In [ ]:
if settings.groq_api_key:
    from groq import Groq
    groq_client = Groq(api_key=settings.groq_api_key)
    
    context_str = " ".join(doc['page_content'] for doc in output.get('documents', []))
    prompt = f"Context: {context_str}\n\nQuestion: What are the benefits of LangGraph?\n\nAnswer concisely based only on context:"
    
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=150
    )
    print("Groq Synthesis Result:")
    print(response.choices[0].message.content)
else:
    print("Live Groq synthesis skipped (GROQ_API_KEY not set). Deterministic offline engine verified.")

### 10. Summary and Architecture Best Practices

In this track, we built production-grade stateful RAG workflows with LangGraph:
1. **CRAG vs Naive DAG**: CRAG replaces blind generation with relevance filtering and adaptive query rewriting.
2. **Self-RAG**: Reflection nodes prevent hallucinations from escaping to users.
3. **Loop Budget Enforcers**: `max_retries` counters prevent infinite loops and runaway billing.
4. **State Persistence**: `MemorySaver` preserves step history and enables pause/resume workflows.
5. **Human-in-the-Loop**: Breakpoints ensure human oversight for critical knowledge tasks.

#### Hands-On Exercises:
1. **Multi-Query Decomposition in Graph**: Add a node before retrieval that splits compound queries into sub-queries.
2. **Web Search Fallback**: Implement a simulated external search node when local retrieval cycles are exhausted.
3. **PostgreSQL Persistence**: Replace `MemorySaver` with `PostgresSaver` for persistent multi-node production deployment.